## Bradford Air Quality Analysis

Reproducible analysis of winter PM2.5 and PM10 air quality data across Bradford monitoring sites, developed for HDRC and councillor briefing

### Configuration

1. **Installing required packages:** 

   This ensures the Python environment has the exact libraries needed to run the analysis.

In [1]:
# Install required packages

import sys
#!{sys.executable} -m pip install -r ../requirements.txt

2. **Import necessary libraries:**

    This loads the tools needed for:
    - file handling + patterns (os, re) 
    - numeric processing (numpy)
    - tabular data analysis (pandas)
    - data visualisation (matplotlib)
    - path creation (Path)

In [2]:
# Import the necessary libraries
from __future__ import annotations

import os
import argparse
import re
from pathlib import Path
from typing import Optional, Tuple, List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

3. **Path allocation:**

    This defines standard folders used in the project for:
    - raw input data (`data/raw`)
    - external data (`data/external`)
    - output directory (`outputs`)

In [3]:
# Create directory structure if it does not already exist
raw_data = Path("../data/raw")
ext_data = Path("../data/external")
out_dir = Path("../outputs")

for path in [raw_data, ext_data, out_dir]:
    path.mkdir(parents=True, exist_ok=True)

4. **Reference thresholds:**

    The following external reference points are used for context only:

    - **PM2.5 (24-hour mean): 15 µg/m³**  
    WHO Global Air Quality Guidelines (2021). 
    - **PM10 (24-hour mean): 50 µg/m³**  
      UK Air Quality Objective used in Local Air Quality Management (LAQM).  

### Data Preprocessing

1. **Load and structure data:** 
    - The raw Excel file uses merged site headers.
    - It loads the data and applies clear column names to enable reliable processing.
    - Also fix the 24:00 o'clock to the next day 00:00 o'clock
    - Drop the invalid rows - especially the summary statistics at the base of the sheet

In [4]:
# Load the sheet
input_file = raw_data / "Multi Station Report 01_11_2024.xlsx"
raw = pd.read_excel(input_file)

# On physical inspection, we only need columns (A to G)
raw = raw.iloc[:, :7].copy()

# Rename columns to a stable schema
raw.columns = [
    "datetime",
    "keighley_pm10", "keighley_pm25",
    "tong_pm10",     "tong_pm25",
    "treadwell_pm10","treadwell_pm25"
]

# Fix 24:00 o'clock, as it is not recognisable as an ideal time 
dt_raw = raw["datetime"].astype(str).str.strip()
mask_24 = dt_raw.str.contains(r"\b24:00\b", na=False)

# Replace 24:00 with 00:00
dt_fixed = dt_raw.copy()
dt_fixed.loc[mask_24] = dt_fixed.loc[mask_24].str.replace("24:00", "00:00", regex=False)

# Parse datetime - UK day-first
dt_parsed = pd.to_datetime(dt_fixed, errors="coerce", dayfirst=True)

# Add one day to rows that originally had 24:00
dt_parsed.loc[mask_24] = dt_parsed.loc[mask_24] + pd.Timedelta(days=1)

raw["datetime"] = dt_parsed

# Now drop invalid rows - especially the summary statistics at the base
raw = raw.dropna(subset=["datetime"]).reset_index(drop=True)

# Coerce numeric pollution columns
for c in raw.columns[1:]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

# A quick peek
display(raw.head())
print("Shape:", raw.shape)

/var/folders/l0/yncq7vg114z8zt1w1r4thfwc0000gn/T/ipykernel_2499/3455288.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt_parsed = pd.to_datetime(dt_fixed, errors="coerce", dayfirst=True)


,datetime,keighley_pm10,keighley_pm25,tong_pm10,tong_pm25,treadwell_pm10,treadwell_pm25
0,2024-11-01 01:00:00,6.2,5.2,10.2,5.8,NaN,NaN
1,2024-11-01 02:00:00,1.0,-0.9,2.1,-1.3,NaN,NaN
2,2024-11-01 03:00:00,6.1,4.6,7.3,4.3,NaN,NaN
3,2024-11-01 04:00:00,5.8,4.2,11.6,7.8,NaN,NaN
4,2024-11-01 05:00:00,2.8,1.0,2.4,-1.4,NaN,NaN


Shape: (1848, 7)


1. **Reshape to tidy format:** 
    - The data is reshaped into a tidy format with one row per site per timestamp  
(`datetime | site | pm10 | pm25`) to support analysis and comparison.

In [5]:
# Reshape to long tidy format: datetime, site, pm10, pm25
site_map = {
    "Bradford Keighley": ("keighley_pm10", "keighley_pm25"),
    "Bradford Tong Street": ("tong_pm10", "tong_pm25"),
    "Bradford Treadwell Mills": ("treadwell_pm10", "treadwell_pm25"),
}

frames = []
for site, (pm10_col, pm25_col) in site_map.items():
    tmp = raw[["datetime", pm10_col, pm25_col]].copy()
    tmp = tmp.rename(columns={pm10_col: "pm10", pm25_col: "pm25"})
    tmp["site"] = site
    frames.append(tmp)

df = pd.concat(frames, ignore_index=True)

# Add time features
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["week"] = df["datetime"].dt.to_period("W").astype(str)

df = df.sort_values(["site", "datetime"]).reset_index(drop=True)

display(df.head())
print("Sites:", df["site"].unique())
print("Datetime range:", df["datetime"].min(), "to", df["datetime"].max())

,datetime,pm10,pm25,site,date,hour,week
0,2024-11-01 01:00:00,6.2,5.2,Bradford Keighley,2024-11-01,1,2024-10-28/2024-11-03
1,2024-11-01 02:00:00,1.0,-0.9,Bradford Keighley,2024-11-01,2,2024-10-28/2024-11-03
2,2024-11-01 03:00:00,6.1,4.6,Bradford Keighley,2024-11-01,3,2024-10-28/2024-11-03
3,2024-11-01 04:00:00,5.8,4.2,Bradford Keighley,2024-11-01,4,2024-10-28/2024-11-03
4,2024-11-01 05:00:00,2.8,1.0,Bradford Keighley,2024-11-01,5,2024-10-28/2024-11-03


Sites: ['Bradford Keighley' 'Bradford Tong Street' 'Bradford Treadwell Mills']
Datetime range: 2024-11-01 01:00:00 to 2025-01-17 00:00:00


In [8]:
# Data Structure
df.shape

(5544, 7)

### Quick QA checks (missingness + hourly coverage)

In [7]:
# -----------------------------
# SENSOR AVAILABILITY (based on non-missing measurements)
# -----------------------------

availability_obs = (
    df.assign(has_data=df[["pm10", "pm25"]].notna().any(axis=1))
      .loc[lambda x: x["has_data"]]
      .groupby("site")["datetime"]
      .agg(
          first_observation="min",
          last_observation="max",
          n_observations="count"
      )
      .reset_index()
)

display(availability_obs)

# Hourly coverage per day (used to identify downtime vs full days)
hours_per_day = (
    df.dropna(subset=["pm10", "pm25"], how="all")
      .groupby(["site", "date"])["hour"]
      .nunique()
      .reset_index(name="hours_reported")
)

display(hours_per_day.describe())


,site,first_observation,last_observation,n_observations
0,Bradford Keighley,2024-11-01 01:00:00,2025-01-17 00:00:00,1828
1,Bradford Tong Street,2024-11-01 01:00:00,2025-01-03 10:00:00,1472
2,Bradford Treadwell Mills,2024-12-21 01:00:00,2025-01-17 00:00:00,648


,hours_reported
count,170.000000
mean,23.223529
std,3.089882
min,1.000000
25%,24.000000
50%,24.000000
75%,24.000000
max,24.000000


In [35]:
# Hours reported per day per site (based on any PM data)
hours_per_day = (
    df.assign(has_data=df[["pm10", "pm25"]].notna().any(axis=1))
      .groupby(["site", "date"], as_index=False)
      .agg(hours_reported=("has_data", "sum"))
)

# Downtime hours assuming an hourly time base
hours_per_day["downtime_hours"] = 24 - hours_per_day["hours_reported"]

# Flag days with any downtime (or choose a stricter threshold like >=6 hours downtime)
hours_per_day["has_downtime"] = hours_per_day["downtime_hours"] > 0

display(hours_per_day.head(20))


,site,date,hours_reported,downtime_hours,has_downtime
0,Bradford Keighley,2024-11-01,23,1,True
1,Bradford Keighley,2024-11-02,24,0,False
2,Bradford Keighley,2024-11-03,24,0,False
3,Bradford Keighley,2024-11-04,24,0,False
4,Bradford Keighley,2024-11-05,24,0,False
5,Bradford Keighley,2024-11-06,24,0,False
6,Bradford Keighley,2024-11-07,24,0,False
7,Bradford Keighley,2024-11-08,24,0,False
8,Bradford Keighley,2024-11-09,24,0,False
9,Bradford Keighley,2024-11-10,24,0,False


In [36]:
downtime_summary = (
    hours_per_day.groupby("site", as_index=False)
                 .agg(
                     days_observed=("date", "nunique"),
                     downtime_days=("has_downtime", "sum"),
                     total_downtime_hours=("downtime_hours", "sum"),
                     avg_hours_reported=("hours_reported", "mean"),
                     min_hours_reported=("hours_reported", "min"),
                 )
)

display(downtime_summary)


,site,days_observed,downtime_days,total_downtime_hours,avg_hours_reported,min_hours_reported
0,Bradford Keighley,78,15,44,23.435897,1
1,Bradford Tong Street,78,30,400,18.871795,0
2,Bradford Treadwell Mills,78,52,1224,8.307692,0


In [37]:
display(
    hours_per_day.sort_values(["downtime_hours"], ascending=False).head(20)
)


,site,date,hours_reported,downtime_hours,has_downtime
168,Bradford Treadwell Mills,2024-11-13,0,24,True
177,Bradford Treadwell Mills,2024-11-22,0,24,True
176,Bradford Treadwell Mills,2024-11-21,0,24,True
175,Bradford Treadwell Mills,2024-11-20,0,24,True
174,Bradford Treadwell Mills,2024-11-19,0,24,True
173,Bradford Treadwell Mills,2024-11-18,0,24,True
172,Bradford Treadwell Mills,2024-11-17,0,24,True
171,Bradford Treadwell Mills,2024-11-16,0,24,True
170,Bradford Treadwell Mills,2024-11-15,0,24,True
169,Bradford Treadwell Mills,2024-11-14,0,24,True


In [30]:
# QA: Missingness + hourly coverage
missing = (
    df.groupby("site")[["pm10", "pm25"]]
      .apply(lambda g: g.isna().mean())
      .rename(columns={"pm10":"pm10_missing_rate", "pm25":"pm25_missing_rate"})
      .reset_index()
)
display(missing)

# Hours per day per site (should be ~24, except period edges or missing data)
hours_per_day = (
    df.dropna(subset=["pm10","pm25"], how="all")
      .groupby(["site", "date"])["hour"]
      .nunique()
      .reset_index(name="unique_hours")
)
display(hours_per_day["unique_hours"].describe())
display(hours_per_day.sort_values("unique_hours").head(10))

,site,pm10_missing_rate,pm25_missing_rate
0,Bradford Keighley,0.029762,0.024892
1,Bradford Tong Street,0.206710,0.221861
2,Bradford Treadwell Mills,0.649351,0.649351


count    170.000000
mean      23.223529
std        3.089882
min        1.000000
25%       24.000000
50%       24.000000
75%       24.000000
max       24.000000
Name: unique_hours, dtype: float64

,site,date,unique_hours
169,Bradford Treadwell Mills,2025-01-17,1
77,Bradford Keighley,2025-01-17,1
136,Bradford Tong Street,2024-12-29,8
141,Bradford Tong Street,2025-01-03,11
137,Bradford Tong Street,2024-12-30,13
135,Bradford Tong Street,2024-12-28,18
34,Bradford Keighley,2024-12-05,20
54,Bradford Keighley,2024-12-25,21
134,Bradford Tong Street,2024-12-27,21
23,Bradford Keighley,2024-11-24,22


In [31]:
# DAILY SUMMARY (per site)
daily = (
    df.groupby(["site", "date"], as_index=False)
      .agg(
          pm10_mean=("pm10", "mean"),
          pm25_mean=("pm25", "mean"),
          pm10_count=("pm10", "count"),
          pm25_count=("pm25", "count"),
      )
)

# Coverage heuristic (>=18 hours = "good coverage day")
daily["pm10_ok_coverage"] = daily["pm10_count"] >= 18
daily["pm25_ok_coverage"] = daily["pm25_count"] >= 18

display(daily.head())
print("Daily rows:", daily.shape[0])


,site,date,pm10_mean,pm25_mean,pm10_count,pm25_count,pm10_ok_coverage,pm25_ok_coverage
0,Bradford Keighley,2024-11-01,11.295652,8.139130,23,23,True,True
1,Bradford Keighley,2024-11-02,14.770833,12.716667,24,24,True,True
2,Bradford Keighley,2024-11-03,17.970833,15.312500,24,24,True,True
3,Bradford Keighley,2024-11-04,24.304167,17.633333,24,24,True,True
4,Bradford Keighley,2024-11-05,34.616667,30.925000,24,24,True,True


Daily rows: 234


In [32]:
# -----------------------------
# SITE SUMMARY + TOP DAYS
# -----------------------------
site_stats = (
    daily.groupby("site", as_index=False)
         .agg(
             pm10_period_mean=("pm10_mean", "mean"),
             pm25_period_mean=("pm25_mean", "mean"),
             pm10_max=("pm10_mean", "max"),
             pm25_max=("pm25_mean", "max"),
             days_observed=("date", "nunique"),
             days_pm10_ok=("pm10_ok_coverage", "sum"),
             days_pm25_ok=("pm25_ok_coverage", "sum"),
         )
).sort_values("pm25_period_mean", ascending=False)

display(site_stats)

top10_pm25 = daily.sort_values("pm25_mean", ascending=False).head(10)
top10_pm10 = daily.sort_values("pm10_mean", ascending=False).head(10)

display(top10_pm25)
display(top10_pm10)


,site,pm10_period_mean,pm25_period_mean,pm10_max,pm25_max,days_observed,days_pm10_ok,days_pm25_ok
2,Bradford Treadwell Mills,16.691311,9.266175,93.037500,30.491667,78,27,27
0,Bradford Keighley,13.618965,9.260507,78.708333,32.516667,78,75,76
1,Bradford Tong Street,13.228413,7.042159,44.421739,31.050000,78,60,61


,site,date,pm10_mean,pm25_mean,pm10_count,pm25_count,pm10_ok_coverage,pm25_ok_coverage
71,Bradford Keighley,2025-01-11,37.870833,32.516667,24,24,True,True
82,Bradford Tong Street,2024-11-05,42.737500,31.050000,24,24,True,True
4,Bradford Keighley,2024-11-05,34.616667,30.925000,24,24,True,True
83,Bradford Tong Street,2024-11-06,44.421739,30.545833,23,24,True,True
232,Bradford Treadwell Mills,2025-01-16,93.037500,30.491667,24,24,True,True
5,Bradford Keighley,2024-11-06,31.845833,28.245833,24,24,True,True
228,Bradford Treadwell Mills,2025-01-12,34.504167,26.458333,24,24,True,True
7,Bradford Keighley,2024-11-08,31.437500,26.304167,24,24,True,True
8,Bradford Keighley,2024-11-09,29.745833,25.991667,24,24,True,True
72,Bradford Keighley,2025-01-12,29.662500,25.941667,24,24,True,True


,site,date,pm10_mean,pm25_mean,pm10_count,pm25_count,pm10_ok_coverage,pm25_ok_coverage
232,Bradford Treadwell Mills,2025-01-16,93.037500,30.491667,24,24,True,True
76,Bradford Keighley,2025-01-16,78.708333,24.704167,24,24,True,True
233,Bradford Treadwell Mills,2025-01-17,62.400000,20.600000,1,1,False,False
83,Bradford Tong Street,2024-11-06,44.421739,30.545833,23,24,True,True
82,Bradford Tong Street,2024-11-05,42.737500,31.050000,24,24,True,True
85,Bradford Tong Street,2024-11-08,41.570833,22.929167,24,24,True,True
71,Bradford Keighley,2025-01-11,37.870833,32.516667,24,24,True,True
4,Bradford Keighley,2024-11-05,34.616667,30.925000,24,24,True,True
228,Bradford Treadwell Mills,2025-01-12,34.504167,26.458333,24,24,True,True
227,Bradford Treadwell Mills,2025-01-11,32.854167,25.420833,24,24,True,True


In [ ]:
# -----------------------------
# LOAD DATA
# -----------------------------
def load_air_quality_excel(path: Path) -> pd.DataFrame:
    """
    Loads the Excel file.
    Tries first sheet; if there are many sheets and the first one is empty, tries others.
    """
    xls = pd.ExcelFile(path)
    for sheet in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        # Keep only if it has some columns and rows
        if df.shape[0] > 0 and df.shape[1] > 0:
            return df
    raise ValueError("Could not find a non-empty sheet in the Excel file.")


raw = load_air_quality_excel(DATA_PATH)

# Standardise column names lightly (preserve originals for detection)
raw.columns = [str(c).strip() for c in raw.columns]

dt_col, site_col, pm25_col, pm10_col = detect_columns(raw)

df = raw.copy()

# Coerce datetime
df["datetime"] = coerce_datetime(df, dt_col)
df["site"] = df[site_col].astype(str).str.strip()

# Coerce pollutant values to numeric
for col, newcol in [(pm25_col, "pm25"), (pm10_col, "pm10")]:
    df[newcol] = pd.to_numeric(df[col], errors="coerce")

# Drop rows with no datetime or site
df = df.dropna(subset=["datetime", "site"]).reset_index(drop=True)

# Add time features
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["weekday"] = df["datetime"].dt.day_name()
df["week"] = df["datetime"].dt.to_period("W").astype(str)

# Sort
df = df.sort_values(["site", "datetime"]).reset_index(drop=True)

# Export cleaned data
clean_csv = OUT_DIR / "cleaned_air_quality_hourly.csv"
df.to_csv(clean_csv, index=False)


# -----------------------------
# QUALITY CHECKS / MISSINGNESS
# -----------------------------
def completeness_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Provide a simple completeness summary by site and pollutant.
    """
    out = []
    for s, g in df.groupby("site"):
        out.append({
            "site": s,
            "rows": len(g),
            "pm25_non_missing": int(g["pm25"].notna().sum()),
            "pm25_missing": int(g["pm25"].isna().sum()),
            "pm10_non_missing": int(g["pm10"].notna().sum()),
            "pm10_missing": int(g["pm10"].isna().sum()),
            "start": g["datetime"].min(),
            "end": g["datetime"].max(),
        })
    return pd.DataFrame(out).sort_values("rows", ascending=False)


qc = completeness_summary(df)
qc_path = OUT_DIR / "qc_completeness_by_site.csv"
qc.to_csv(qc_path, index=False)


# -----------------------------
# DAILY AGGREGATION
# -----------------------------
daily = (
    df.groupby(["site", "date"], as_index=False)
      .agg(
          pm25_mean=("pm25", "mean"),
          pm10_mean=("pm10", "mean"),
          pm25_count=("pm25", "count"),
          pm10_count=("pm10", "count"),
          datetime_min=("datetime", "min"),
          datetime_max=("datetime", "max"),
      )
)

# Helpful flags: "enough coverage" (e.g., >= 18 hours of data)
daily["pm25_ok_coverage"] = daily["pm25_count"] >= 18
daily["pm10_ok_coverage"] = daily["pm10_count"] >= 18

# Exceedance flags (contextual)
daily["pm25_exceeds_who_24h"] = daily["pm25_mean"] > WHO_PM25_24H
daily["pm10_exceeds_uk_daily_obj"] = daily["pm10_mean"] > UK_PM10_DAILY_OBJ

daily_path = OUT_DIR / "daily_summary_by_site.csv"
daily.to_csv(daily_path, index=False)


# -----------------------------
# INSIGHTS TABLES
# -----------------------------
# 1) Overall period stats by site
site_stats = (
    daily.groupby("site", as_index=False)
         .agg(
             pm25_period_mean=("pm25_mean", "mean"),
             pm10_period_mean=("pm10_mean", "mean"),
             pm25_max=("pm25_mean", "max"),
             pm10_max=("pm10_mean", "max"),
             days_observed=("date", "nunique"),
             days_pm25_ok=("pm25_ok_coverage", "sum"),
             days_pm10_ok=("pm10_ok_coverage", "sum"),
             days_pm25_who_exceed=("pm25_exceeds_who_24h", "sum"),
             days_pm10_uk_exceed=("pm10_exceeds_uk_daily_obj", "sum"),
         )
)

site_stats = site_stats.sort_values("pm25_period_mean", ascending=False)
site_stats_path = OUT_DIR / "site_level_period_stats.csv"
site_stats.to_csv(site_stats_path, index=False)

# 2) Top 10 worst days overall (by PM2.5 and PM10)
top_pm25_days = (
    daily.dropna(subset=["pm25_mean"])
         .sort_values("pm25_mean", ascending=False)
         .head(10)
         .reset_index(drop=True)
)
top_pm10_days = (
    daily.dropna(subset=["pm10_mean"])
         .sort_values("pm10_mean", ascending=False)
         .head(10)
         .reset_index(drop=True)
)

top_pm25_path = OUT_DIR / "top10_days_by_pm25.csv"
top_pm10_path = OUT_DIR / "top10_days_by_pm10.csv"
top_pm25_days.to_csv(top_pm25_path, index=False)
top_pm10_days.to_csv(top_pm10_path, index=False)

# 3) Weekly smoothing (useful for councillor-friendly trend)
weekly = (
    daily.groupby(["site", "week"], as_index=False)
         .agg(pm25_week_mean=("pm25_mean", "mean"),
              pm10_week_mean=("pm10_mean", "mean"),
              pm25_who_exceed_days=("pm25_exceeds_who_24h", "sum"),
              pm10_uk_exceed_days=("pm10_exceeds_uk_daily_obj", "sum"))
)
weekly_path = OUT_DIR / "weekly_summary_by_site.csv"
weekly.to_csv(weekly_path, index=False)


# -----------------------------
# PLOTS (Matplotlib) — saves PNGs
# -----------------------------
def save_plot(fig: plt.Figure, filename: str) -> None:
    out_path = OUT_DIR / filename
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


# Plot 1: Overall daily trend (all sites averaged)
overall_daily = (
    daily.groupby("date", as_index=False)
         .agg(pm25_mean=("pm25_mean", "mean"),
              pm10_mean=("pm10_mean", "mean"))
)
overall_daily["date"] = pd.to_datetime(overall_daily["date"])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(overall_daily["date"], overall_daily["pm25_mean"], label="PM2.5 (daily mean, across sites)")
ax.plot(overall_daily["date"], overall_daily["pm10_mean"], label="PM10 (daily mean, across sites)")
ax.axhline(WHO_PM25_24H, linestyle="--", linewidth=1, label="WHO PM2.5 24h guideline (15 µg/m³)")
ax.axhline(UK_PM10_DAILY_OBJ, linestyle="--", linewidth=1, label="UK PM10 daily objective (50 µg/m³)")
ax.set_title("Bradford winter period: overall PM2.5 and PM10 (daily mean across sites)")
ax.set_xlabel("Date")
ax.set_ylabel("µg/m³")
ax.legend()
save_plot(fig, "plot1_overall_daily_trend.png")


# Plot 2: Trend by site (PM2.5) — small multiples via separate lines
sites = sorted(daily["site"].unique())
fig, ax = plt.subplots(figsize=(11, 5))
for s in sites:
    sdat = daily.loc[daily["site"] == s].copy()
    sdat["date"] = pd.to_datetime(sdat["date"])
    ax.plot(sdat["date"], sdat["pm25_mean"], label=s)
ax.axhline(WHO_PM25_24H, linestyle="--", linewidth=1)
ax.set_title("PM2.5 by site (daily mean)")
ax.set_xlabel("Date")
ax.set_ylabel("µg/m³")
ax.legend(ncol=2, fontsize=8)
save_plot(fig, "plot2_pm25_by_site_daily.png")


# Plot 3: Trend by site (PM10)
fig, ax = plt.subplots(figsize=(11, 5))
for s in sites:
    sdat = daily.loc[daily["site"] == s].copy()
    sdat["date"] = pd.to_datetime(sdat["date"])
    ax.plot(sdat["date"], sdat["pm10_mean"], label=s)
ax.axhline(UK_PM10_DAILY_OBJ, linestyle="--", linewidth=1)
ax.set_title("PM10 by site (daily mean)")
ax.set_xlabel("Date")
ax.set_ylabel("µg/m³")
ax.legend(ncol=2, fontsize=8)
save_plot(fig, "plot3_pm10_by_site_daily.png")


# Plot 4: Site ranking by average PM2.5
rank = site_stats.sort_values("pm25_period_mean", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(rank["site"], rank["pm25_period_mean"])
ax.axvline(WHO_PM25_24H, linestyle="--", linewidth=1)
ax.set_title("Site ranking: average PM2.5 over the period")
ax.set_xlabel("PM2.5 (µg/m³)")
ax.set_ylabel("Site")
save_plot(fig, "plot4_site_ranking_pm25.png")


# Plot 5: Exceedance days by site (WHO PM2.5 24h)
ex = site_stats.sort_values("days_pm25_who_exceed", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ex["site"], ex["days_pm25_who_exceed"])
ax.set_title("Days exceeding WHO PM2.5 24h guideline (15 µg/m³) by site")
ax.set_xlabel("Number of days (within period)")
ax.set_ylabel("Site")
save_plot(fig, "plot5_pm25_who_exceed_days_by_site.png")


# Plot 6: Weekly smoothed PM2.5 and PM10 (overall)
overall_weekly = (
    weekly.groupby("week", as_index=False)
          .agg(pm25=("pm25_week_mean", "mean"),
               pm10=("pm10_week_mean", "mean"))
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(overall_weekly["week"], overall_weekly["pm25"], label="PM2.5 (weekly mean across sites)")
ax.plot(overall_weekly["week"], overall_weekly["pm10"], label="PM10 (weekly mean across sites)")
ax.axhline(WHO_PM25_24H, linestyle="--", linewidth=1)
ax.axhline(UK_PM10_DAILY_OBJ, linestyle="--", linewidth=1)
ax.set_title("Weekly smoothed PM2.5 and PM10 (across sites)")
ax.set_xlabel("Week")
ax.set_ylabel("µg/m³")
ax.legend()
plt.xticks(rotation=45, ha="right")
save_plot(fig, "plot6_overall_weekly_trend.png")


# -----------------------------
# EXPORT ONE EXCEL PACK (handy for review)
# -----------------------------
excel_pack = OUT_DIR / "bradford_air_quality_analysis_pack.xlsx"
with pd.ExcelWriter(excel_pack, engine="openpyxl") as writer:
    qc.to_excel(writer, sheet_name="QC_Completeness", index=False)
    daily.to_excel(writer, sheet_name="Daily_By_Site", index=False)
    site_stats.to_excel(writer, sheet_name="Site_Stats", index=False)
    top_pm25_days.to_excel(writer, sheet_name="Top10_PM25_Days", index=False)
    top_pm10_days.to_excel(writer, sheet_name="Top10_PM10_Days", index=False)
    weekly.to_excel(writer, sheet_name="Weekly_By_Site", index=False)
    overall_daily.to_excel(writer, sheet_name="Overall_Daily", index=False)


# -----------------------------
# AUTO-GENERATE A SLIDE OUTLINE TEXT (optional helper)
# -----------------------------
slide_outline = f"""
SLIDE OUTLINE (5–10 mins)

1) Purpose
- Councillors want to understand winter air quality across Bradford sites (PM2.5, PM10).

2) Data
- Period: {df['datetime'].min().date()} to {df['datetime'].max().date()}
- Sites: {df['site'].nunique()} monitoring sites
- Metrics: PM2.5, PM10 (µg/m³)
- Approach: daily means for clarity; site-level comparison retained.

3) Key findings: overall trends
- Use plot1_overall_daily_trend.png
- Mention any peak periods found in Top10 tables.

4) Key findings: differences by site
- Use plot4_site_ranking_pm25.png
- Use plot2_pm25_by_site_daily.png and/or plot3_pm10_by_site_daily.png

5) Health context
- WHO PM2.5 24h reference: {WHO_PM25_24H} µg/m³ (shown as dashed line)
- Optional UK PM10 daily objective: {UK_PM10_DAILY_OBJ} µg/m³

6) Interpretation
- Winter drivers: heating, traffic, cold-air inversions; local geography.

7) Limitations
- No meteorology/traffic/deprivation in the extract; short time window; missingness varies by site.

8) If more time
- Add weather + traffic covariates; link deprivation; longer historic comparisons; intervention targeting.

Outputs saved to: {OUT_DIR}
"""
(OUT_DIR / "slide_outline.txt").write_text(slide_outline.strip(), encoding="utf-8")


print("DONE ✅")
print(f"Outputs folder: {OUT_DIR}")
print(f"Cleaned CSV: {clean_csv}")
print(f"Excel pack: {excel_pack}")
print("Saved plots:")
for p in sorted(OUT_DIR.glob("plot*.png")):
    print(" -", p.name)
print("Slide outline:", OUT_DIR / "slide_outline.txt")
